<a href="https://colab.research.google.com/github/Ritvik2090/loan-portfolio-analytics/blob/main/notebooks/02_payment_processor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Loading Libraries and Data

In [50]:
import pandas as pd
import numpy as np

from datetime import datetime

pd.set_option("display.max_columns", None)

In [52]:
from google.colab import files

uploaded = files.upload()

Saving customers.csv to customers.csv
Saving emis.csv to emis.csv
Saving loan_products.csv to loan_products.csv
Saving loans.csv to loans.csv
Saving payment_events.csv to payment_events.csv


In [53]:
loans = pd.read_csv("loans.csv")
emis = pd.read_csv("emis.csv")
payment_events = pd.read_csv("payment_events.csv")
loan_products = pd.read_csv("loan_products.csv")

In [54]:
date_cols = [
    "due_date",
    "created_at",
    "updated_at"
]

for col in date_cols:
    emis[col] = pd.to_datetime(emis[col])

event_dates = [
    "payment_date",
    "created_at",
    "updated_at"
]

for col in event_dates:
    payment_events[col] = pd.to_datetime(payment_events[col])

# Copy of Emi with helper Columns

In [55]:
emis_work = emis.copy()

emis_work["principal_remaining"] = emis_work["principal_amount"]

emis_work["interest_remaining"] = emis_work["interest_amount"]

emis_work["status"] = "PENDING"

In [56]:
emis_work = (
    emis_work
    .sort_values(
        ["loan_id", "emi_number"]
    )
    .reset_index(drop=True)
)

payment_events = (
    payment_events
    .sort_values(
        ["loan_id", "payment_date"]
    )
    .reset_index(drop=True)
)

In [57]:
print(emis_work.head())

print()

print(payment_events.head())

   id  loan_id  emi_number   due_date  emi_amount  principal_amount  \
0   1        1           1 2025-09-26        3418              3188   
1   2        1           2 2025-10-26        3418              3220   
2   3        1           3 2025-11-26        3418              3253   
3   4        1           4 2025-12-26        3418              3285   
4   5        1           5 2026-01-26        3418              3318   

   interest_amount   status created_at updated_at  principal_remaining  \
0              230  PENDING 2025-08-29 2026-07-23                 3188   
1              198  PENDING 2025-08-29 2026-07-23                 3220   
2              165  PENDING 2025-08-29 2026-07-23                 3253   
3              133  PENDING 2025-08-29 2026-07-23                 3285   
4              100  PENDING 2025-08-29 2026-07-23                 3318   

   interest_remaining  
0                 230  
1                 198  
2                 165  
3                 133  
4       

# Next Pending EMI

In [58]:
def get_next_pending_emi(loan_emis):

    pending = loan_emis[
        loan_emis["status"] != "PAID"
    ]

    if pending.empty:
        return None

    return pending.index[0]

In [59]:
loan1 = emis_work[
    emis_work["loan_id"] == 1
]

idx = get_next_pending_emi(loan1)

print(idx)
print(emis_work.loc[idx])

0
id                                       1
loan_id                                  1
emi_number                               1
due_date               2025-09-26 00:00:00
emi_amount                            3418
principal_amount                      3188
interest_amount                        230
status                             PENDING
created_at             2025-08-29 00:00:00
updated_at             2026-07-23 00:00:00
principal_remaining                   3188
interest_remaining                     230
Name: 0, dtype: object


# Allocation of amount to EMIs

In [60]:
def allocate_to_emi(
    emis_work,
    idx,
    payment_amount
):

    interest_remaining = emis_work.at[idx, "interest_remaining"]
    principal_remaining = emis_work.at[idx, "principal_remaining"]

    interest_amount = emis_work.at[idx, "interest_amount"]
    principal_amount = emis_work.at[idx, "principal_amount"]

    # -----------------------------
    # Interest First
    # -----------------------------

    interest_paid = min(
        payment_amount,
        interest_remaining
    )

    payment_amount -= interest_paid
    interest_remaining -= interest_paid

    # -----------------------------
    # Principal Next
    # -----------------------------

    principal_paid = min(
        payment_amount,
        principal_remaining
    )

    payment_amount -= principal_paid
    principal_remaining -= principal_paid

    # -----------------------------
    # Update dataframe
    # -----------------------------

    emis_work.at[idx, "interest_remaining"] = interest_remaining
    emis_work.at[idx, "principal_remaining"] = principal_remaining

    if (
        interest_remaining == 0
        and
        principal_remaining == 0
    ):
        status = "PAID"

    elif (
        interest_remaining < interest_amount
        or
        principal_remaining < principal_amount
    ):
        status = "PARTIAL"

    else:
        status = "PENDING"

    emis_work.at[idx, "status"] = status

    return {
        "interest_paid": interest_paid,
        "principal_paid": principal_paid,
        "remaining_amount": payment_amount
    }

# Populating Payments

In [61]:
payment_id = 1

def create_payment_row(
    payment_event,
    emis_work,
    idx,
    allocation
):

    global payment_id

    payment = {

        "payment_id": payment_id,

        "payment_event_id": payment_event["id"],

        "loan_id": payment_event["loan_id"],

        "emi_id": emis_work.at[idx, "id"],

        "payment_date": payment_event["payment_date"],

        "payment_amount":
            allocation["interest_paid"]
            + allocation["principal_paid"],

        "principal_paid":
            allocation["principal_paid"],

        "interest_paid":
            allocation["interest_paid"],

        "payment_mode":
            payment_event["payment_mode"],

        "created_at":
            payment_event["created_at"],

        "updated_at":
            payment_event["updated_at"]
    }

    payment_id += 1

    return payment

# Processing the payment events

In [62]:
payments = []

def process_payment_event(payment_event):

    global payments

    remaining_amount = payment_event["paid_amount"]

    loan_id = payment_event["loan_id"]

    while remaining_amount > 0:

        loan_emis = emis_work[
            emis_work["loan_id"] == loan_id
        ]

        idx = get_next_pending_emi(loan_emis)

        if idx is None:
            break

        allocation = allocate_to_emi(
            emis_work,
            idx,
            remaining_amount
        )

        payment = create_payment_row(
            payment_event,
            emis_work,
            idx,
            allocation
        )

        payments.append(payment)

        remaining_amount = allocation["remaining_amount"]

In [63]:
payments = []
payment_id = 1

emis_work = emis.copy()

emis_work["principal_remaining"] = emis_work["principal_amount"]
emis_work["interest_remaining"] = emis_work["interest_amount"]
emis_work["status"] = "PENDING"

In [64]:
for _, event in payment_events.iterrows():
    process_payment_event(event)

payments_df = pd.DataFrame(payments)

print("Processing Complete")
print("Payment Events :", len(payment_events))
print("Payment Rows   :", len(payments_df))

Processing Complete
Payment Events : 200121
Payment Rows   : 202777


# Verification

In [65]:
print("Payment Events :", len(payment_events))
print(
    "Unique Events Processed :",
    payments_df["payment_event_id"].nunique()
)

Payment Events : 200121
Unique Events Processed : 200093


In [66]:
print(
    "Payment Event Total :",
    payment_events["paid_amount"].sum()
)

print(
    "Payments Total :",
    payments_df["payment_amount"].sum()
)

print(
    "Difference :",
    payment_events["paid_amount"].sum()
    -
    payments_df["payment_amount"].sum()
)

Payment Event Total : 4195849377
Payments Total : 4195849377
Difference : 0


In [67]:
processed = set(payments_df["payment_event_id"].unique())
all_events = set(payment_events["id"])

missing = all_events - processed

print(len(missing))
print(sorted(list(missing))[:30])

28
[6695, 50609, 52387, 64942, 74776, 81180, 83889, 86658, 90163, 108265, 116211, 118552, 131669, 131853, 135234, 139561, 141776, 144706, 151056, 161649, 166954, 169249, 174900, 177094, 178634, 181868, 193671, 198184]


In [73]:
payment_events[
    payment_events["id"].isin(missing)
].sort_values(["loan_id","payment_date"])

,id,loan_id,paid_amount,payment_date,payment_mode,created_at,updated_at
6694,6695,496,0,2025-10-20,Net Banking,2026-07-23,2026-07-23
50608,50609,3757,0,2024-06-01,UPI,2026-07-23,2026-07-23
52386,52387,3890,0,2026-02-20,Net Banking,2026-07-23,2026-07-23
64941,64942,4849,0,2025-07-07,Net Banking,2026-07-23,2026-07-23
74775,74776,5583,0,2023-12-26,Bank Transfer,2026-07-23,2026-07-23
81179,81180,6045,0,2024-05-25,Bank Transfer,2026-07-23,2026-07-23
83888,83889,6250,0,2025-04-08,UPI,2026-07-23,2026-07-23
86657,86658,6467,0,2024-06-16,Net Banking,2026-07-23,2026-07-23
90162,90163,6732,0,2023-06-05,UPI,2026-07-23,2026-07-23
108264,108265,8104,0,2024-02-02,UPI,2026-07-23,2026-07-23


In [74]:
non_zero_events = payment_events[payment_events["paid_amount"] > 0]

print("Non-zero payment events :", len(non_zero_events))
print("Payment rows generated  :", len(payments_df))

Non-zero payment events : 200093
Payment rows generated  : 202777


In [80]:
processed = payments_df["payment_event_id"].nunique()

print(processed)

200093


In [81]:
# 1
assert (
    payments_df["payment_amount"]
    ==
    payments_df["principal_paid"]
    +
    payments_df["interest_paid"]
).all()

# 2
assert (
    payment_events["paid_amount"].sum()
    ==
    payments_df["payment_amount"].sum()
)

# 3
assert (emis_work["principal_remaining"] >= 0).all()

# 4
assert (emis_work["interest_remaining"] >= 0).all()

# 5
assert (
    payments_df["payment_event_id"].nunique()
    ==
    len(payment_events[payment_events["paid_amount"] > 0])
)

print("✅ ALL PORTFOLIO VALIDATIONS PASSED")

✅ ALL PORTFOLIO VALIDATIONS PASSED


In [82]:
emis_work.head()

,id,loan_id,emi_number,due_date,emi_amount,principal_amount,interest_amount,status,created_at,updated_at,principal_remaining,interest_remaining
0,1,1,1,2025-09-26,3418,3188,230,PAID,2025-08-29,2026-07-23,0,0
1,2,1,2,2025-10-26,3418,3220,198,PAID,2025-08-29,2026-07-23,0,0
2,3,1,3,2025-11-26,3418,3253,165,PAID,2025-08-29,2026-07-23,0,0
3,4,1,4,2025-12-26,3418,3285,133,PAID,2025-08-29,2026-07-23,0,0
4,5,1,5,2026-01-26,3418,3318,100,PAID,2025-08-29,2026-07-23,0,0


# Final EMI and Payments Table

In [83]:
# Production-style EMI schedule
final_emis = emis_work.drop(
    columns=["principal_remaining", "interest_remaining"]
)
final_emis.to_csv("emis.csv", index=False)

# Debug/working version
emis_work.to_csv("emis_work.csv", index=False)

# Payment allocations
payments_df.to_csv("payments.csv", index=False)